HC subtype1 subtype2 BNA246脑区mean INT 差异比较
箱体填充风格

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.conftest import matplotlib
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import statsmodels.api as sm
matplotlib.use('Agg')
# =========================================================
# 1. 读取文件与数据合并
# =========================================================
base_dir = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode/Step2_BN246mean_INT/'
# 注意：请确保以下读取的csv文件中都包含 'mean_fd' 列
df_hc = pd.read_csv(base_dir + 'HC_mean_INT_agesex_FD.csv')
df_s1 = pd.read_csv(base_dir + 'subtype1_mean_INT_agesex_FD.csv')
df_s2 = pd.read_csv(base_dir + 'subtype2_mean_INT_agesex_FD.csv')

df_hc['group'] = 'HC'
df_s1['group'] = 'Subtype1'
df_s2['group'] = 'Subtype2'

df = pd.concat([df_hc, df_s1, df_s2], ignore_index=True)

# 【修改点 1】：在提取列时，加入 'mean_fd' 并去除缺失值
df = df[['group', 'age', 'sex', 'mean_INT', 'mean_fd']].dropna()

# =========================================================
# 2. ANCOVA 主效应模型构建
# =========================================================
# 【修改点 2】：将 mean_fd 添加到线性回归模型公式中
model = ols('mean_INT ~ C(group) + age + C(sex) + mean_fd', data=df).fit()

# 计算协方差分析表 (主效应) 并保存
anova_table = sm.stats.anova_lm(model, typ=2)
print('========== 协方差分析表 (ANCOVA 主效应) ==========')
print(anova_table)
anova_table.to_csv(
    base_dir + 'Diff_mean_INT_ANCOVA_MainEffect_covFD.csv'
)  # 更新文件名以示区别

# =========================================================
# 3. 事后检验分析 (Pairwise Comparisons) 与 FDR 校正
# =========================================
pairwise_results = []
# 这里的 contrast 不变，因为我们只关心 C(group) 的差异，其他协变量 (age, sex, mean_fd) 在对比时保持不变（互相抵消）
contrasts = {
    'HC_vs_Subtype1': "C(group)[T.Subtype1] = 0",
    'HC_vs_Subtype2': "C(group)[T.Subtype2] = 0",
    'Subtype1_vs_Subtype2': 'C(group)[T.Subtype1] - C(group)[T.Subtype2] = 0',
}

for comp_name, contrast_str in contrasts.items():
  t_test_res = model.t_test(contrast_str)
  p_val = (
      t_test_res.pvalue.item()
      if hasattr(t_test_res.pvalue, 'item')
      else float(t_test_res.pvalue)
  )
  t_val = (
      t_test_res.tvalue.item()
      if hasattr(t_test_res.tvalue, 'item')
      else float(t_test_res.tvalue)
  )
  pairwise_results.append(
      {'Comparison': comp_name, 't_value': t_val, 'p_uncorrected': p_val}
  )

pairwise_df = pd.DataFrame(pairwise_results)

# 对这 3 次两两比较进行 FDR 多重比较校正
_, p_fdr_pair, _, _ = multipletests(
    pairwise_df['p_uncorrected'], alpha=0.05, method='fdr_bh'
)
pairwise_df['p_FDR'] = p_fdr_pair


def get_stars(p):
  if p < 0.001:
    return '***'
  elif p < 0.01:
    return '**'
  elif p < 0.05:
    return '*'
  else:
    return 'ns'


pairwise_df['Significance'] = pairwise_df['p_FDR'].apply(get_stars)

# 保存事后检验详细结果
pairwise_df.to_csv(
    base_dir + 'Diff_mean_INT_ANCOVA_Pairwise_covFD.csv', index=False
)
print('\n========== 事后两两比较 (FDR 校正) ==========')
print(pairwise_df)


# =========================================================
# 4. 绘制并动态智能标注显著性的箱体图
# ==========================================
plt.figure(figsize=(8, 6))

# 【修改点】设置画板背景为白色，边框加粗
sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})

# 【修改点】自定义配色字典
custom_palette = {'HC': '#42B8B1', 'Subtype1': '#B50A27', 'Subtype2': '#00478D'}

# 绘制箱体图
ax = sns.boxplot(
    x='group',
    y='mean_INT',
    data=df,
    order=['HC', 'Subtype1', 'Subtype2'],
    palette=custom_palette,
    width=0.5,
    showfliers=False,
    linewidth=1.7,
    boxprops={'alpha': 0.8, 'edgecolor': '#3C3B3D'}, # #3C3B3D
)
# groups_order = ['HC', 'Subtype1', 'Subtype2']
# # 【修复点】：使用 enumerate 提取索引 i，解决原代码中字符串相乘报错的问题
# for i, group_name in enumerate(groups_order):
#     group_color = '#3C3B3D'
#     group_color = custom_palette[group_name]
#     # boxplot 生成的线条包含 whiskers(须), caps(盖帽), medians(中位数线)
#     ax.lines[i*5 + 0].set_color(group_color)  # whisker 1
#     ax.lines[i*5 + 1].set_color(group_color)  # whisker 2
#     ax.lines[i*5 + 2].set_color(group_color)  # cap 1
#     ax.lines[i*5 + 3].set_color(group_color)  # cap 2
#     ax.lines[i*5 + 4].set_color(group_color)  # median linemedian line
# 绘制散点图
sns.stripplot(
    x='group',
    y='mean_INT',
    data=df,
    order=['HC', 'Subtype1', 'Subtype2'],
    palette=custom_palette,
    alpha=0.5,
    size=6,
    jitter=True,
)

# 设置四周边框（上下左右轴）为黑色，线条加粗
for spine in ax.spines.values():
  spine.set_edgecolor('black')
  spine.set_linewidth(1.7)
  spine.set_visible(True)

y_max = df['mean_INT'].max()
y_min = df['mean_INT'].min()
y_range = y_max - y_min

# 连线的动态高度步长和拐角小尾巴长度
step = 0.08 * y_range
h = 0.02 * y_range
current_y = y_max + 0.05 * y_range

# 提取 FDR 校正后的星号
star_hc_s1 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype1'][
    'Significance'
].values[0]
star_s1_s2 = pairwise_df[pairwise_df['Comparison'] == 'Subtype1_vs_Subtype2'][
    'Significance'
].values[0]
star_hc_s2 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype2'][
    'Significance'
].values[0]


def add_bracket(x1, x2, y, text_star):
  plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, color='black')
  plt.text(
      (x1 + x2) * 0.5,
      y + h + 0.01 * y_range,
      text_star,
      ha='center',
      va='bottom',
      color='black',
      fontsize=14,
      fontweight='bold',
  )

# 动态堆叠画线逻辑：只有显著的 ('ns' 以外) 才画线
if star_hc_s1 != 'ns':
  add_bracket(0, 1, current_y, star_hc_s1)
  current_y += step

if star_s1_s2 != 'ns':
  add_bracket(1, 2, current_y, star_s1_s2)
  current_y += step

if star_hc_s2 != 'ns':
  add_bracket(0, 2, current_y, star_hc_s2)
  current_y += step

# 自动调整 y 轴范围，为顶部的连线留出空间
plt.ylim(y_min - 0.05 * y_range, current_y + 0.1 * y_range)

# 图表标签与字体大小调整
plt.ylabel('', fontsize=14)
plt.xlabel('', fontsize=14)
plt.xticks(fontsize=26)
plt.yticks(fontsize=26)
plt.tight_layout()

# 保存并展示图片
img_save_path = base_dir + 'Fig_Diff_mean_INT_ANCOVA_Stars_covFD2.png'
plt.savefig(img_save_path, dpi=300, bbox_inches='tight')

Copy on Write disabled


========== 协方差分析表 (ANCOVA 主效应) ==========
             sum_sq     df          F        PR(>F)
C(group)   9.424423    2.0  75.603074  2.465436e-28
C(sex)     0.433970    1.0   6.962641  8.667940e-03
age        0.024847    1.0   0.398643  5.281734e-01
mean_fd    0.377047    1.0   6.049373  1.435943e-02
Residual  23.497771  377.0        NaN           NaN

========== 事后两两比较 (FDR 校正) ==========
             Comparison    t_value  p_uncorrected         p_FDR Significance
0        HC_vs_Subtype1   6.722794   6.615250e-11  6.615250e-11          ***
1        HC_vs_Subtype2  -8.002430   1.522108e-14  2.283162e-14          ***
2  Subtype1_vs_Subtype2  12.156125   6.482904e-29  1.944871e-28          ***


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_84957/4178539429.py:111: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_84957/4178539429.py:134: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


HC subtype1 subtype2 BNA246脑区mean INT 差异比较
箱体不填充风格

In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib
matplotlib.use('Agg')

base_dir = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step2_BN246mean_INT/'
df_hc = pd.read_csv(base_dir + 'HC_mean_INT_agesex_FD.csv')
df_s1 = pd.read_csv(base_dir + 'subtype1_mean_INT_agesex_FD.csv')
df_s2 = pd.read_csv(base_dir + 'subtype2_mean_INT_agesex_FD.csv')

df_hc['group'] = 'HC'
df_s1['group'] = 'Subtype1'
df_s2['group'] = 'Subtype2'
df = pd.concat([df_hc, df_s1, df_s2], ignore_index=True)
df = df[['group', 'age', 'sex', 'mean_INT', 'mean_fd']].dropna()

# ANCOVA
model = ols('mean_INT ~ C(group) + age + C(sex) + mean_fd', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print('========== 协方差分析表 (ANCOVA 主效应) ==========')
print(anova_table)
anova_table.to_csv(base_dir + 'Diff_mean_INT_ANCOVA_MainEffect_covFD.csv')

# Group主效应
F_group = anova_table.loc['C(group)', 'F']
p_group = anova_table.loc['C(group)', 'PR(>F)']
df_group = anova_table.loc['C(group)', 'df']
df_residual = anova_table.loc['Residual', 'df']

overall_results = pd.DataFrame({
    'Effect': ['Group'],
    'df1': [df_group],
    'df2': [df_residual],
    'F_value': [F_group],
    'p_uncorrected': [p_group]
})
overall_results.to_csv(base_dir + 'Diff_mean_INT_ANCOVA_OverallResult_covFD.csv', index=False)
print(f'\n========== Overall Group Effect ==========\nGroup: F({int(df_group)}, {int(df_residual)}) = {F_group:.4f}, p = {p_group:.6g}')

# 两两比较
contrasts = {
    'HC_vs_Subtype1': "C(group)[T.Subtype1] = 0",
    'HC_vs_Subtype2': "C(group)[T.Subtype2] = 0",
    'Subtype1_vs_Subtype2': 'C(group)[T.Subtype1] - C(group)[T.Subtype2] = 0',
}
pairwise_results = []
for comp_name, contrast_str in contrasts.items():
    t_test_res = model.t_test(contrast_str)
    p_val = t_test_res.pvalue.item() if hasattr(t_test_res.pvalue, 'item') else float(t_test_res.pvalue)
    t_val = t_test_res.tvalue.item() if hasattr(t_test_res.tvalue, 'item') else float(t_test_res.tvalue)
    pairwise_results.append({'Comparison': comp_name, 't_value': t_val, 'p_uncorrected': p_val})

pairwise_df = pd.DataFrame(pairwise_results)
_, p_fdr_pair, _, _ = multipletests(pairwise_df['p_uncorrected'], alpha=0.05, method='fdr_bh')
pairwise_df['p_FDR'] = p_fdr_pair

def get_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'
pairwise_df['Significance'] = pairwise_df['p_FDR'].apply(get_stars)
pairwise_df.to_csv(base_dir + 'Diff_mean_INT_ANCOVA_Pairwise_covFD.csv', index=False)
print('\n========== 事后两两比较 (FDR 校正) ==========')
print(pairwise_df)

# 汇总结果
summary_rows = []
summary_rows.append({
    'Analysis': 'Overall ANCOVA',
    'Comparison': 'HC vs Subtype1 vs Subtype2',
    'Statistic': 'F',
    'df1': df_group,
    'df2': df_residual,
    'Statistic_value': F_group,
    'p_uncorrected': p_group,
    'p_FDR': np.nan,
    'Significance': get_stars(p_group)
})
for _, row in pairwise_df.iterrows():
    summary_rows.append({
        'Analysis': 'Pairwise ANCOVA',
        'Comparison': row['Comparison'],
        'Statistic': 't',
        'df1': np.nan,
        'df2': df_residual,
        'Statistic_value': row['t_value'],
        'p_uncorrected': row['p_uncorrected'],
        'p_FDR': row['p_FDR'],
        'Significance': row['Significance']
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(base_dir + 'Diff_mean_INT_ANCOVA_AllResults_covFD.csv', index=False)
print('\n========== Overall + Pairwise Summary ==========')
print(summary_df)

# 绘图
plt.figure(figsize=(8, 6))
sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})
custom_palette = {'HC': '#9a9da1', 'Subtype1': '#B50A27', 'Subtype2': '#00478D'}
groups_order = ['HC', 'Subtype1', 'Subtype2']

ax = sns.boxplot(
    x='group', y='mean_INT', data=df, order=groups_order, palette=custom_palette,
    width=0.5, showfliers=False, linewidth=1.7, boxprops={'facecolor': 'none', 'alpha': 0.8}
)
for i, group_name in enumerate(groups_order):
    group_color = custom_palette[group_name]
    if i < len(ax.patches):
        ax.patches[i].set_facecolor('none')
        ax.patches[i].set_edgecolor(group_color)
        ax.patches[i].set_linewidth(3)
    for j in range(5):
        line_idx = i * 5 + j
        if line_idx < len(ax.lines):
            ax.lines[line_idx].set_color(group_color)
            ax.lines[line_idx].set_linewidth(4)

sns.stripplot(
    x='group', y='mean_INT', data=df, order=groups_order, palette=custom_palette,
    alpha=0.8, size=6, jitter=True
)

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(3.0)
    spine.set_visible(True)

y_max, y_min = df['mean_INT'].max(), df['mean_INT'].min()
y_range = y_max - y_min
step = 0.08 * y_range
h = 0.02 * y_range
current_y = y_max + 0.05 * y_range

star_hc_s1 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype1']['Significance'].values[0]
star_s1_s2 = pairwise_df[pairwise_df['Comparison'] == 'Subtype1_vs_Subtype2']['Significance'].values[0]
star_hc_s2 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype2']['Significance'].values[0]

def add_bracket(x1, x2, y, text_star):
    plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=2.5, color='black')
    plt.text((x1 + x2) * 0.5, y, text_star, ha='center', va='bottom', color='black', fontsize=18, fontweight='bold')

if star_hc_s1 != 'ns':
    add_bracket(0, 1, current_y, star_hc_s1)
    current_y += step
if star_s1_s2 != 'ns':
    add_bracket(1, 2, current_y, star_s1_s2)
    current_y += step
if star_hc_s2 != 'ns':
    add_bracket(0, 2, current_y, star_hc_s2)
    current_y += step

plt.ylim(y_min - 0.05 * y_range, current_y + 0.1 * y_range)
plt.ylabel('', fontsize=14)
plt.xlabel('', fontsize=14)
plt.xticks(fontsize=26)
plt.yticks(fontsize=26)
plt.tight_layout()

img_save_path = base_dir + 'Fig_Diff_mean_INT_ANCOVA_Stars_covFD.png'
plt.savefig(img_save_path, dpi=300, bbox_inches='tight')
plt.close()



========== 协方差分析表 (ANCOVA 主效应) ==========
             sum_sq     df          F        PR(>F)
C(group)   6.917062    2.0  56.596892  4.240573e-22
C(sex)     0.439967    1.0   7.199808  7.627635e-03
age        0.013914    1.0   0.227698  6.335259e-01
mean_fd    0.626658    1.0  10.254904  1.484659e-03
Residual  21.998933  360.0        NaN           NaN

========== Overall Group Effect ==========
Group: F(2, 360) = 56.5969, p = 4.24057e-22

========== 事后两两比较 (FDR 校正) ==========
             Comparison    t_value  p_uncorrected         p_FDR Significance
0        HC_vs_Subtype1   4.463956   1.077902e-05  1.077902e-05          ***
1        HC_vs_Subtype2  -8.169017   5.327284e-15  7.990926e-15          ***
2  Subtype1_vs_Subtype2  10.257054   7.950169e-22  2.385051e-21          ***

========== Overall + Pairwise Summary ==========
          Analysis                  Comparison Statistic  df1    df2  \
0   Overall ANCOVA  HC vs Subtype1 vs Subtype2         F  2.0  360.0   
1  Pairwise ANCOV

/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2614324354.py:108: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2614324354.py:124: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


HC subtype1 subtype2  8各网络平均 INT  差异比较
按照 2x4 布局  （箱体不进行颜色填充）

In [14]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines

path = "/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new"

files = {
    'HC': path + '/HC_INT20_7net_agesex_FD.csv',
    'Subtype1': path + '/subtype1_INT20_7net_agesex.csv',
    'Subtype2': path + '/subtype2_INT20_7net_agesex.csv'
}

networks = ['subcortical', 'Visual', 'Somatomotor', 'Dorsal_Attention', 'Ventral_Attention', 'Limbic', 'Frontoparietal', 'Default']
network_labels = ['SC', 'VIS', 'SMN', 'DAN', 'VAN', 'LIM', 'FPN', 'DMN']

dfs = []
for group, file in files.items():
    temp = pd.read_csv(file)
    temp['group'] = group
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df = df[['group', 'age', 'sex', 'mean_fd'] + networks].dropna()
print(f"Total N = {len(df)} (HC = {(df.group == 'HC').sum()}, Subtype1 = {(df.group == 'Subtype1').sum()}, Subtype2 = {(df.group == 'Subtype2').sum()})")

contrasts = {
    'HC_vs_Subtype1': 'C(group)[T.Subtype1] = 0',
    'HC_vs_Subtype2': 'C(group)[T.Subtype2] = 0',
    'Subtype1_vs_Subtype2': 'C(group)[T.Subtype1] - C(group)[T.Subtype2] = 0'
}

overall_results = []
pairwise_results = []
for net in networks:
    model = ols(f'{net} ~ C(group) + age + C(sex) + mean_fd', data=df).fit()
    anova = sm.stats.anova_lm(model, typ=2)
    overall_results.append({
        'Network': net,
        'F_value': anova.loc['C(group)', 'F'],
        'df1': anova.loc['C(group)', 'df'],
        'df2': anova.loc['Residual', 'df'],
        'p_uncorrected': anova.loc['C(group)', 'PR(>F)']
    })
    for comparison, contrast in contrasts.items():
        test = model.t_test(contrast)
        pairwise_results.append({
            'Network': net,
            'Comparison': comparison,
            't_value': float(test.tvalue),
            'p_uncorrected': float(test.pvalue)
        })

overall_df = pd.DataFrame(overall_results)
pairwise_df = pd.DataFrame(pairwise_results)
overall_df['p_FDR'] = multipletests(overall_df['p_uncorrected'], method='fdr_bh')[1]
pairwise_df['p_FDR'] = multipletests(pairwise_df['p_uncorrected'], method='fdr_bh')[1]

def get_stars(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'

overall_df['Significance'] = overall_df['p_FDR'].apply(get_stars)
pairwise_df['Significance'] = pairwise_df['p_FDR'].apply(get_stars)

name_map = dict(zip(networks, network_labels))
overall_df['Network_label'] = overall_df['Network'].map(name_map)
pairwise_df['Network_label'] = pairwise_df['Network'].map(name_map)

overall_df = overall_df[['Network', 'Network_label', 'F_value', 'df1', 'df2', 'p_uncorrected', 'p_FDR', 'Significance']]
pairwise_df = pairwise_df[['Network', 'Network_label', 'Comparison', 't_value', 'p_uncorrected', 'p_FDR', 'Significance']]

result_dir = os.path.join(path, 'Step3_INT7Net_Diff')
os.makedirs(result_dir, exist_ok=True)
overall_csv = os.path.join(result_dir, 'INT_8Network_ANCOVA_Overall.csv')
pairwise_csv = os.path.join(result_dir, 'INT_8Network_ANCOVA_Pairwise.csv')
overall_df.to_csv(overall_csv, index=False, encoding='utf-8-sig')
pairwise_df.to_csv(pairwise_csv, index=False, encoding='utf-8-sig')

overall_summary = overall_df.rename(columns={
    'F_value': 'Overall_F',
    'p_uncorrected': 'Overall_p',
    'p_FDR': 'Overall_q',
    'Significance': 'Overall_Significance'
})
pairwise_wide = pairwise_df.pivot(index=['Network', 'Network_label'], columns='Comparison', values=['t_value', 'p_uncorrected', 'p_FDR', 'Significance'])
pairwise_wide.columns = [f'{metric}_{comparison}' for metric, comparison in pairwise_wide.columns]
summary_df = overall_summary.merge(pairwise_wide.reset_index(), on=['Network', 'Network_label'], how='left')
summary_csv = os.path.join(result_dir, 'INT_8Network_ANCOVA_AllResults.csv')
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')

print('\n=== Overall ANCOVA ===')
print(overall_df[['Network_label', 'F_value', 'df1', 'df2', 'p_uncorrected', 'p_FDR', 'Significance']].to_string(index=False))
print('\n=== Pairwise comparisons ===')
print(pairwise_df[['Network_label', 'Comparison', 't_value', 'p_uncorrected', 'p_FDR', 'Significance']].to_string(index=False))
print('\nResults saved:')
print(overall_csv)
print(pairwise_csv)
print(summary_csv)

corrected_data = df.copy()
for net in networks:
    model = ols(f'{net} ~ age + C(sex) + mean_fd', data=df).fit()
    corrected_data[net] = model.resid + df[net].mean()

all_values = corrected_data[networks].values.flatten()
all_values = all_values[~pd.isna(all_values)]
y_min = all_values.min()
y_max = all_values.max()
y_range = y_max - y_min
h = y_range * 0.035
step = y_range * 0.12
ylim_bottom = y_min - y_range * 0.05
ylim_top = y_max + step * 3.5

group_order = ['HC', 'Subtype1', 'Subtype2']
palette = {'HC': '#9a9da1', 'Subtype1': '#B50A27', 'Subtype2': '#00478D'}
sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (net, label) in enumerate(zip(networks, network_labels)):
    ax = axes[i]
    plot_df = corrected_data[['group', net]].rename(columns={net: 'INT'}).dropna()
    sns.boxplot(x='group', y='INT', data=plot_df, order=group_order, palette=palette, width=0.5, showfliers=False, linewidth=2, ax=ax)
    colors = [palette[g] for g in group_order]
    for j, patch in enumerate(ax.patches):
        patch.set_facecolor('none')
        patch.set_edgecolor(colors[j % 3])
    for j, artist in enumerate(ax.artists):
        artist.set_facecolor('none')
        artist.set_edgecolor(colors[j % 3])
    if len(ax.lines) and len(ax.lines) % 3 == 0:
        lines_per_box = len(ax.lines) // 3
        for j, line in enumerate(ax.lines):
            line.set_color(colors[j // lines_per_box])
            line.set_linewidth(2)
    sns.stripplot(x='group', y='INT', data=plot_df, order=group_order, alpha=0.4, jitter=True, size=5, palette=palette, legend=False, ax=ax)
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(ylim_bottom, ylim_top)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(label, fontsize=18, fontweight='bold', pad=10)
    ax.set_xlabel('')
    if i % 4 == 0:
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_linewidth(4)
        ax.set_ylabel('INT Value', fontsize=22, fontweight='bold')
        ax.tick_params(axis='y', labelsize=20, width=2.5, length=6)
    else:
        ax.spines['left'].set_visible(False)
        ax.set_ylabel('')
        ax.tick_params(axis='y', left=False, labelleft=False)
    if i >= 4:
        ax.spines['bottom'].set_visible(True)
        ax.spines['bottom'].set_linewidth(4)
        ax.set_xticklabels(['HC', 'Subtype1', 'Subtype2'], fontsize=18)
        ax.tick_params(axis='x', width=2.5, length=6)
    else:
        ax.spines['bottom'].set_visible(False)
        ax.set_xticklabels([])
        ax.tick_params(axis='x', bottom=False, labelbottom=False)

    comparisons = [('HC_vs_Subtype1', 0, 1), ('Subtype1_vs_Subtype2', 1, 2), ('HC_vs_Subtype2', 0, 2)]
    current_y = y_max + h
    for comparison, x1, x2 in comparisons:
        star = pairwise_df.loc[(pairwise_df['Network'] == net) & (pairwise_df['Comparison'] == comparison), 'Significance'].iloc[0]
        if star != 'ns':
            ax.plot([x1, x1, x2, x2], [current_y, current_y + h, current_y + h, current_y], lw=2.5, color='black')
            ax.text((x1 + x2) / 2, current_y + h, star, ha='center', va='bottom', fontsize=16, fontweight='bold')
            current_y += step

legend_handles = [mlines.Line2D([], [], color=palette[group], marker='s', linestyle='None', markersize=14, label=group) for group in group_order]
fig.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(0.96, 0.98), fontsize=16, frameon=False, ncol=3, handletextpad=0.3)
plt.subplots_adjust(hspace=0.15, wspace=0.08)
out_fig = os.path.join(result_dir, 'Fig_Subtype_8NetINT_Diff_2x4_Hollow.png')
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
plt.show()
print(f'\nFigure saved to: {out_fig}')


Total N = 366 (HC = 215, Subtype1 = 67, Subtype2 = 84)

=== Overall ANCOVA ===
Network_label   F_value  df1   df2  p_uncorrected        p_FDR Significance
           SC 30.588079  2.0 360.0   5.382325e-13 6.151228e-13          ***
          VIS 37.816812  2.0 360.0   1.237655e-15 1.980248e-15          ***
          SMN 36.619409  2.0 360.0   3.338298e-15 4.451064e-15          ***
          DAN 54.337775  2.0 360.0   2.384715e-21 1.907772e-20          ***
          VAN 42.638573  2.0 360.0   2.404237e-17 4.808474e-17          ***
          LIM 13.623543  2.0 360.0   1.979620e-06 1.979620e-06          ***
          FPN 48.916663  2.0 360.0   1.611082e-19 4.296220e-19          ***
          DMN 52.631524  2.0 360.0   8.886059e-21 3.554423e-20          ***

=== Pairwise comparisons ===
Network_label           Comparison   t_value  p_uncorrected        p_FDR Significance
           SC       HC_vs_Subtype1  2.838661   4.787270e-03 4.787270e-03           **
           SC       HC_vs_Subtype2 

/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2433913790.py:54: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  't_value': float(test.tvalue),
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2433913790.py:54: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  't_value': float(test.tvalue),
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2433913790.py:54: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  't_value': float(tes


Figure saved to: /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step3_INT7Net_Diff/Fig_Subtype_8NetINT_Diff_2x4_Hollow.png


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2433913790.py:184: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Variance 画图

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import zscore
from nilearn.conftest import matplotlib
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# 如果不需要在独立窗口弹出图片，可以使用 Agg 后端
matplotlib.use('Agg')

# 统一定义输出基础路径，方便后续拼写文件名
base_dir = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step4_a_INTVariance/'

# =========================================================
# 1. 读取文件与数据合并
# =========================================================
df_hc = pd.read_csv(base_dir + 'HC_INT_7net_variance.csv')
df_s1 = pd.read_csv(base_dir + 'subtype1_INT_7net_variance.csv')
df_s2 = pd.read_csv(base_dir + 'subtype2_INT_7net_variance.csv')

# 提取需要的列 (假设 HC 文件里的 age/sex 列名已经是 age, sex)
df_hc = df_hc[['subID', 'mean_fd', 'age', 'sex', 'variance']]
df_s1 = df_s1[['subID', 'mean_fd', 'age', 'sex', 'variance']]
df_s2 = df_s2[['subID', 'mean_fd', 'age', 'sex', 'variance']]

df_hc['group'] = 'HC'
df_s1['group'] = 'Subtype1'
df_s2['group'] = 'Subtype2'

# 合并得到原始全量数据
df_raw = pd.concat([df_hc, df_s1, df_s2], ignore_index=True)
df_raw = df_raw[['group', 'age', 'sex', 'variance', 'mean_fd']].dropna()

# =========================================================
# 2. 异常值剔除处理 (按 group 分组计算 > 3 SD)
# =========================================================
# 按组计算 Z-score (自由度 ddof=1)
df_raw['variance_zscore'] = df_raw.groupby('group')['variance'].transform(lambda x: zscore(x, ddof=1))

# 设定 3 个标准差为阈值
threshold = 3.0

# 提取绝对值小于等于 3 的数据作为清洗后数据
df_clean = df_raw[np.abs(df_raw['variance_zscore']) <= threshold].copy()

print("========== 数据异常值检查 ==========")
print(f"原始数据样本量: {len(df_raw)}")
print(f"清洗后数据样本量: {len(df_clean)}")
print(f"剔除的异常值数量: {len(df_raw) - len(df_clean)}")
print("====================================\n")


# =========================================================
# 3. 核心统计与绘图函数封装
# =========================================================
def run_analysis_and_plot(df, suffix, title_note):
    """
    执行 ANCOVA, 事后两两比较, 并绘制箱线图
    :param df: 输入的数据框 (df_raw 或 df_clean)
    :param suffix: 输出文件名的后缀 ('Raw' 或 'Clean')
    :param title_note: 图片标题的额外注释 (如果不需要显示标题可在下方代码中注销)
    """
    print(f"开始处理: 【{suffix}】 组数据分析")

    # 1. ANCOVA 主效应模型构建
    model = ols('variance ~ C(group) + age + C(sex) + mean_fd', data=df).fit()

    # 计算协方差分析表 (主效应) 并保存
    anova_table = sm.stats.anova_lm(model, typ=2)
    anova_csv_path = base_dir + f'Diff_variance_ANCOVA_MainEffect_covFD_{suffix}.csv'
    anova_table.to_csv(anova_csv_path)

    # 2. 事后检验分析 (Pairwise Comparisons) 与 FDR 校正
    pairwise_results = []
    contrasts = {
        'HC_vs_Subtype1': 'C(group)[T.Subtype1] = 0',
        'HC_vs_Subtype2': 'C(group)[T.Subtype2] = 0',
        'Subtype1_vs_Subtype2': 'C(group)[T.Subtype1] - C(group)[T.Subtype2] = 0'
    }

    for comp_name, contrast_str in contrasts.items():
        t_test_res = model.t_test(contrast_str)
        p_val = t_test_res.pvalue.item() if hasattr(t_test_res.pvalue, 'item') else float(t_test_res.pvalue)
        t_val = t_test_res.tvalue.item() if hasattr(t_test_res.tvalue, 'item') else float(t_test_res.tvalue)
        pairwise_results.append({
            'Comparison': comp_name,
            't_value': t_val,
            'p_uncorrected': p_val
        })

    pairwise_df = pd.DataFrame(pairwise_results)

    # FDR 校正
    _, p_fdr_pair, _, _ = multipletests(pairwise_df['p_uncorrected'], alpha=0.05, method='fdr_bh')
    pairwise_df['p_FDR'] = p_fdr_pair

    def get_stars(p):
        if p < 0.001:
            return '***'
        elif p < 0.01:
            return '**'
        elif p < 0.05:
            return '*'
        else:
            return 'ns'

    pairwise_df['Significance'] = pairwise_df['p_FDR'].apply(get_stars)
    pairwise_csv_path = base_dir + f'Diff_variance_ANCOVA_Pairwise_covFD_{suffix}.csv'
    #pairwise_df.to_csv(pairwise_csv_path, index=False)

    print(f"[{suffix}] 事后两两比较 (FDR 校正) 结果:")
    print(pairwise_df)
    print("-" * 40)

    # =========================================================
    # 3. 绘制箱体填充为空白无色、线条赋指定颜色的箱体图
    # =========================================================
    plt.figure(figsize=(8, 6))

    # 设置画板背景为白色
    sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})

    # 自定义配色字典
    custom_palette = {'HC': '#9a9da1', 'Subtype1': '#B50A27', 'Subtype2': '#00478D'}
    groups_order = ['HC', 'Subtype1', 'Subtype2']

    # 绘制箱线图
    ax = sns.boxplot(
        x='group',
        y='variance',
        data=df,
        order=groups_order,
        palette=custom_palette,
        width=0.5,
        showfliers=False,
        linewidth=1.7,
        boxprops={'facecolor': 'none', 'alpha': 0.8}
    )

    # 将箱体的外框、须线、盖帽、中位数线统一修改为对应组的专属配色
    for i, group_name in enumerate(groups_order):
        group_color = custom_palette[group_name]

        # 1. 对应箱体自身的边框（box 边缘）
        if i < len(ax.patches):
            ax.patches[i].set_facecolor('none')       # 确保箱体为空白无色填充
            ax.patches[i].set_edgecolor(group_color)  # 将外框线修改为专属颜色
            ax.patches[i].set_linewidth(3)

        # 兼容部分版本（补充判断）
        if hasattr(ax, 'artists') and i < len(ax.artists):
            ax.artists[i].set_facecolor('none')
            ax.artists[i].set_edgecolor(group_color)
            ax.artists[i].set_linewidth(3)

        # 2. 对应箱体的内部线条（whiskers, caps, medians 依次为每组 5 条线）
        for j in range(5):
            line_idx = i * 5 + j
            if line_idx < len(ax.lines):
                ax.lines[line_idx].set_color(group_color)
                ax.lines[line_idx].set_linewidth(4)

    # 绘制散点图
    sns.stripplot(
        x='group',
        y='variance',
        data=df,
        order=groups_order,
        palette=custom_palette,
        alpha=0.8,
        size=6,
        jitter=True,
        ax=ax
    )

    # 设置四周边框（上下左右轴）为黑色，线条加粗
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(3.0)
        spine.set_visible(True)

    y_max = df['variance'].max()
    y_min = df['variance'].min()
    y_range = y_max - y_min

    # 连线的动态高度步长和拐角小尾巴长度
    step = 0.08 * y_range
    h = 0.02 * y_range
    current_y = y_max + 0.05 * y_range

    star_hc_s1 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype1']['Significance'].values[0]
    star_s1_s2 = pairwise_df[pairwise_df['Comparison'] == 'Subtype1_vs_Subtype2']['Significance'].values[0]
    star_hc_s2 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype2']['Significance'].values[0]

    def add_bracket(x1, x2, y, text_star):
        plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=2.5, color='black')
        # 按照提供的代码代理，此处文本高度置于 y
        plt.text((x1 + x2) * 0.5, y, text_star, ha='center', va='bottom', color='black', fontsize=18, fontweight='bold')

    # 动态堆叠画线逻辑：只有显著的 ('ns' 以外) 才画线
    if star_hc_s1 != 'ns':
        add_bracket(0, 1, current_y, star_hc_s1)
        current_y += step

    if star_s1_s2 != 'ns':
        add_bracket(1, 2, current_y, star_s1_s2)
        current_y += step

    if star_hc_s2 != 'ns':
        add_bracket(0, 2, current_y, star_hc_s2)
        current_y += step

    # 自动调整 y 轴范围，为顶部的连线留出空间
    plt.ylim(y_min - 0.05 * y_range, current_y + 0.1 * y_range)

    # 图表标签与字体大小调整
    plt.ylabel('', fontsize=14)
    plt.xlabel('', fontsize=14)
    plt.xticks(fontsize=26)

    # ----------------------------------------------------
    # 这里修改：针对剔除异常值后的图片，定制 Y 轴刻度
    # ----------------------------------------------------
    if suffix == 'Clean':
        plt.yticks([0.0, 0.1, 0.2, 0.3], fontsize=26)
    else:
        plt.yticks(fontsize=26)
    # ----------------------------------------------------

    # 增加 title 注释，如果不需要可以注释掉这行
    #plt.title(f'{title_note}', fontsize=16, pad=20)

    plt.tight_layout()

    # 保存并展示图片
    img_save_path = base_dir + f'Fig_Diff_variance_ANCOVA_Stars_covFD_{suffix}.png'
    plt.savefig(img_save_path, dpi=300, bbox_inches='tight')
    plt.close()  # 必须关闭当前图，否则两张图会叠加在同一个画布上

    print(f"=== [{suffix}] 绘图完成！图片已保存至: {img_save_path} ===\n")


# =========================================================
# 4. 运行分析
# =========================================================
# 第一次运行：原始数据 (Raw)
run_analysis_and_plot(df_raw, suffix='Raw', title_note='[Original Data]')

# 第二次运行：剔除 >3SD 后的数据 (Clean)
run_analysis_and_plot(df_clean, suffix='Clean', title_note='[Outliers > 3SD Removed]')

print("所有步骤执行完毕！两套结果（原始的与剔除异常值的）均已分别保存。")

========== 数据异常值检查 ==========
原始数据样本量: 366
清洗后数据样本量: 363
剔除的异常值数量: 3

开始处理: 【Raw】 组数据分析
[Raw] 事后两两比较 (FDR 校正) 结果:
             Comparison   t_value  p_uncorrected     p_FDR Significance
0        HC_vs_Subtype1  2.460412       0.014347  0.014347            *
1        HC_vs_Subtype2 -3.466505       0.000591  0.000887          ***
2  Subtype1_vs_Subtype2  4.835852       0.000002  0.000006          ***
----------------------------------------


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2476792601.py:131: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2476792601.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


=== [Raw] 绘图完成！图片已保存至: /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step4_a_INTVariance/Fig_Diff_variance_ANCOVA_Stars_covFD_Raw.png ===

开始处理: 【Clean】 组数据分析
[Clean] 事后两两比较 (FDR 校正) 结果:
             Comparison   t_value  p_uncorrected         p_FDR Significance
0        HC_vs_Subtype1  3.111460   2.011618e-03  2.011618e-03           **
1        HC_vs_Subtype2 -4.093321   5.262457e-05  7.893685e-05          ***
2  Subtype1_vs_Subtype2  5.887220   9.063076e-09  2.718923e-08          ***
----------------------------------------


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2476792601.py:131: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2476792601.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


=== [Clean] 绘图完成！图片已保存至: /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step4_a_INTVariance/Fig_Diff_variance_ANCOVA_Stars_covFD_Clean.png ===

所有步骤执行完毕！两套结果（原始的与剔除异常值的）均已分别保存。


Range 基于 Network的 range

In [16]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import zscore
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# 若不需要在本地直接弹窗，可以使用 Agg 后端
matplotlib.use('Agg')

# ==========================================
# 1. 读取数据
# ==========================================
# 建议修改为您实际的本地 base_dir 路径
base_dir = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step4_b_INTRange/'
df_hc = pd.read_csv(base_dir + 'HC_INT8Net_Range.csv')
df1 = pd.read_csv(base_dir + 'subtype1_BN246INT_Range.csv')
df2 = pd.read_csv(base_dir + 'subtype2_BN246INT_Range.csv')

# ==========================================
# 2. 添加组别标签并合并数据
# ==========================================
df_hc['group'] = 'HC'
df1['group'] = 'Subtype1'
df2['group'] = 'Subtype2'

# 将三组数据合并，并删除含有缺失值的行
df_raw = pd.concat([df_hc, df1, df2], ignore_index=True)
df_raw = df_raw.dropna(subset=['difference', 'age', 'sex', 'mean_fd'])

# 为了在模型中有明确的对比基准，我们将 group 转换为有顺序的分类变量
df_raw['group'] = pd.Categorical(df_raw['group'], categories=['HC', 'Subtype1', 'Subtype2'])

# =========================================================
# 3. 异常值剔除处理 (按 group 分组计算 > 3 SD)
# =========================================================
# 按组计算 Z-score (自由度 ddof=1)
df_raw['difference_zscore'] = df_raw.groupby('group')['difference'].transform(lambda x: zscore(x, ddof=1))

# 设定 3 个标准差为阈值
threshold = 3.0

# 提取绝对值小于等于 3 的数据作为清洗后数据
df_clean = df_raw[np.abs(df_raw['difference_zscore']) <= threshold].copy()

print("========== 数据异常值检查 ==========")
print(f"原始数据样本量: {len(df_raw)}")
print(f"清洗后数据样本量: {len(df_clean)}")
print(f"剔除的异常值数量: {len(df_raw) - len(df_clean)}")
print("====================================\n")


# =========================================================
# 4. 核心统计与绘图函数封装
# =========================================================
def run_analysis_and_plot(df_input, suffix):
    print(f"\n开始处理: 【{suffix}】 组数据分析")

    # --- 统计检验：协方差分析 (ANCOVA) 主效应 ---
    model = ols('difference ~ C(group, Treatment("HC")) + age + C(sex) + mean_fd', data=df_input).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"[{suffix}] ANCOVA 主效应统计结果:")
    print(anova_table)

    # 保存主效应结果
    anova_table.to_csv(base_dir + f'Diff_difference_ANCOVA_MainEffect_covFD_{suffix}.csv')

    # --- 事后检验 (Pairwise Comparisons) 与 FDR 校正 ---
    pairwise_results = []
    contrasts = {
        'HC_vs_Subtype1': 'C(group, Treatment("HC"))[T.Subtype1] = 0',
        'HC_vs_Subtype2': 'C(group, Treatment("HC"))[T.Subtype2] = 0',
        'Subtype1_vs_Subtype2': 'C(group, Treatment("HC"))[T.Subtype1] - C(group, Treatment("HC"))[T.Subtype2] = 0'
    }

    for comp_name, contrast_str in contrasts.items():
        t_test_res = model.t_test(contrast_str)
        p_val = t_test_res.pvalue.item() if hasattr(t_test_res.pvalue, 'item') else float(t_test_res.pvalue)
        t_val = t_test_res.tvalue.item() if hasattr(t_test_res.tvalue, 'item') else float(t_test_res.tvalue)
        pairwise_results.append({
            'Comparison': comp_name,
            't_value': t_val,
            'p_uncorrected': p_val
        })

    pairwise_df = pd.DataFrame(pairwise_results)

    # FDR 校正
    _, p_fdr, _, _ = multipletests(pairwise_df['p_uncorrected'], alpha=0.05, method='fdr_bh')
    pairwise_df['p_FDR'] = p_fdr

    def get_stars(p):
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return 'ns'

    pairwise_df['stars'] = pairwise_df['p_FDR'].apply(get_stars)
    print(f"[{suffix}] 事后两两比较 (FDR 校正):")
    print(pairwise_df)

    # 保存事后比较结果
    pairwise_df.to_csv(base_dir + f'Diff_difference_ANCOVA_Pairwise_covFD_{suffix}.csv', index=False)

    # --- 绘制箱体填充为空白无色、线条赋指定颜色的箱体图 ---
    plt.figure(figsize=(8, 6))
    sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})

    custom_palette = {'HC': '#9a9da1', 'Subtype1': '#B50A27', 'Subtype2': '#00478D'}
    groups_order = ['HC', 'Subtype1', 'Subtype2']

    ax = sns.boxplot(
        x='group',
        y='difference',
        data=df_input,
        order=groups_order,
        palette=custom_palette,
        width=0.5,
        showfliers=False,
        linewidth=1.7,
        boxprops={'facecolor': 'none', 'alpha': 0.8}
    )

    for i, group_name in enumerate(groups_order):
        group_color = custom_palette[group_name]

        if i < len(ax.patches):
            ax.patches[i].set_facecolor('none')
            ax.patches[i].set_edgecolor(group_color)
            ax.patches[i].set_linewidth(3)

        if hasattr(ax, 'artists') and i < len(ax.artists):
            ax.artists[i].set_facecolor('none')
            ax.artists[i].set_edgecolor(group_color)
            ax.artists[i].set_linewidth(3)

        for j in range(5):
            line_idx = i * 5 + j
            if line_idx < len(ax.lines):
                ax.lines[line_idx].set_color(group_color)
                ax.lines[line_idx].set_linewidth(4)

    sns.stripplot(
        x='group',
        y='difference',
        data=df_input,
        order=groups_order,
        palette=custom_palette,
        alpha=0.8,
        size=6,
        jitter=True,
        ax=ax
    )

    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(3.0)
        spine.set_visible(True)

    y_max = df_input['difference'].max()
    y_min = df_input['difference'].min()
    y_range = y_max - y_min

    step = 0.08 * y_range
    h = 0.02 * y_range
    current_y = y_max + 0.05 * y_range

    star_hc_s1 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype1']['stars'].values[0]
    star_s1_s2 = pairwise_df[pairwise_df['Comparison'] == 'Subtype1_vs_Subtype2']['stars'].values[0]
    star_hc_s2 = pairwise_df[pairwise_df['Comparison'] == 'HC_vs_Subtype2']['stars'].values[0]

    def add_bracket(x1, x2, y, text_star):
        plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=2.5, color='black')
        plt.text((x1 + x2) * 0.5, y, text_star, ha='center', va='bottom', color='black', fontsize=18, fontweight='bold')

    if star_hc_s1 != 'ns':
        add_bracket(0, 1, current_y, star_hc_s1)
        current_y += step

    if star_s1_s2 != 'ns':
        add_bracket(1, 2, current_y, star_s1_s2)
        current_y += step

    if star_hc_s2 != 'ns':
        add_bracket(0, 2, current_y, star_hc_s2)
        current_y += step

    plt.ylim(y_min - 0.05 * y_range, current_y + 0.1 * y_range)

    plt.ylabel('', fontsize=20, fontweight='bold')
    plt.xlabel('', fontsize=14)
    plt.xticks(fontsize=26)

    # ====================================================
    # 根据 Suffix 动态调整 Y 轴的刻度数值
    # ====================================================
    if suffix == 'Clean':
        plt.yticks([0.5, 1.0, 1.5,2.0], fontsize=24)
    elif suffix == 'Raw':
        plt.yticks([0.5, 1.5, 2.5], fontsize=24)
    else:
        plt.yticks(fontsize=24)
    # ====================================================

    plt.tight_layout()

    save_path = base_dir + f'Fig_Range_ANCOVA_{suffix}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"绘图完成！已保存至 {save_path}")

# =========================================================
# 5. 执行分析
# =========================================================
# 第一次运行：未处理的原始数据
run_analysis_and_plot(df_raw, suffix='Raw')

# 第二次运行：剔除了 >3SD 的干净数据
run_analysis_and_plot(df_clean, suffix='Clean')

print("\n所有步骤执行完毕！两套结果（原始的与剔除异常值的）均已分别保存。")

/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2789886567.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_raw['difference_zscore'] = df_raw.groupby('group')['difference'].transform(lambda x: zscore(x, ddof=1))
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2789886567.py:115: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2789886567.py:146: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


========== 数据异常值检查 ==========
原始数据样本量: 366
清洗后数据样本量: 364
剔除的异常值数量: 2


开始处理: 【Raw】 组数据分析
[Raw] ANCOVA 主效应统计结果:
                              sum_sq     df          F    PR(>F)
C(group, Treatment("HC"))   1.282812    2.0  13.040922  0.000003
C(sex)                      0.002085    1.0   0.042389  0.836996
age                         0.002372    1.0   0.048225  0.826306
mean_fd                     0.454563    1.0   9.242075  0.002538
Residual                   17.706277  360.0        NaN       NaN
[Raw] 事后两两比较 (FDR 校正):
             Comparison   t_value  p_uncorrected     p_FDR stars
0        HC_vs_Subtype1  2.638818   8.680960e-03  0.008681    **
1        HC_vs_Subtype2 -3.535947   4.593881e-04  0.000689   ***
2  Subtype1_vs_Subtype2  5.042647   7.291565e-07  0.000002   ***
绘图完成！已保存至 /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step4_b_INTRange/Fig_Range_ANCOVA_Raw.png

开始处理: 【Clean】 组数据分析
[Clean] ANCOVA 主效应统计结果:
                              sum_sq

/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2789886567.py:115: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/2789886567.py:146: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


亚型间大的层级差异的差异


In [17]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.colors import LinearSegmentedColormap  # 新增导入

# ----------------- 0. 基础路径和名称设置 -----------------
base_dir = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/'
out_dir = os.path.join(base_dir, 'Step5_DiffNetwork')
if not os.path.exists(out_dir):
    os.makedirs(out_dir)  # 如果输出文件夹不存在则创建

file1 = os.path.join(base_dir, 'subtype1_INT20_7net_agesex.csv')
file2 = os.path.join(base_dir, 'subtype2_INT20_7net_agesex.csv')

ordered_networks = [
    'Default', 'Frontoparietal', 'Limbic', 'Ventral_Attention',
    'Dorsal_Attention', 'Somatomotor', 'Visual', 'subcortical'
]

# ----------------- 1. 读取数据并计算 差异矩阵 -----------------
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

data1 = df1[ordered_networks].values
data2 = df2[ordered_networks].values

# 计算  差异矩阵
diff_1 = data1[:, :, np.newaxis] - data1[:, np.newaxis, :]
diff_2 = data2[:, :, np.newaxis] - data2[:, np.newaxis, :]

# ----------------- 2. 提取上三角数据，拉平并与协变量对应保存 -----------------
# 提取上三角索引 (k=1 表示不包含主对角线)，共有 8*7/2 = 28 个连接
upper_tri_indices = np.triu_indices(8, k=1)

# 生成这 28 个连接的名称，例如 "Default-Frontoparietal"
edge_names = [f"{ordered_networks[i]}-{ordered_networks[j]}" for i, j in zip(*upper_tri_indices)]


def flatten_and_merge(df_orig, diff_3d, group_label):
    # 提取所有被试的上三角数据，形状变为 (N, 28)
    flattened_edges = diff_3d[:, upper_tri_indices[0], upper_tri_indices[1]]

    # 转换为 DataFrame
    df_edges = pd.DataFrame(flattened_edges, columns=edge_names)

    # 提取协变量
    df_covariates = df_orig[['subID', 'age', 'sex', 'mean_fd']].copy()
    df_covariates['Group'] = group_label  # 添加组别标签 (1代表Subtype1, 0代表Subtype2)

    # 拼接并返回
    return pd.concat([df_covariates, df_edges], axis=1)


# 处理两组数据并保存
df1_flat = flatten_and_merge(df1, diff_1, group_label=1)
df2_flat = flatten_and_merge(df2, diff_2, group_label=0)

df1_flat.to_csv(os.path.join(out_dir, 'subtype1_edges_flattened.csv'), index=False)
df2_flat.to_csv(os.path.join(out_dir, 'subtype2_edges_flattened.csv'), index=False)
print("需求1已完成：单被试的 28 个网络差异已拉平并与协变量合并保存。")

# ----------------- 3. 控制协变量的组间比较 (OLS回归) -----------------
# 合并两组数据用于统计分析
df_all = pd.concat([df1_flat, df2_flat], ignore_index=True)

# 定义自变量 X (组别 + 年龄 + 性别 + meanFD)
# 注意：我们要看的是 Group 的主效应，Subtype1(1) vs Subtype2(0) 意味着 T为正则 Subtype1的差值更大
X = df_all[['Group', 'age', 'sex', 'mean_fd']]
X = sm.add_constant(X)  # 添加常数项(截距)

t_vals_28 = []
p_vals_28 = []

# 对 28 个连接逐一进行多元线性回归
for edge in edge_names:
    y = df_all[edge]
    model = sm.OLS(y, X).fit()
    # 提取 'Group' 这一项的 t值 和 p值
    t_vals_28.append(model.tvalues['Group'])
    p_vals_28.append(model.pvalues['Group'])

# ----------------- 4. p值校正与 8x8 矩阵还原 -----------------
# 使用 Benjamini/Hochberg 方法对 28 个 P 值进行 FDR 校正
reject, p_vals_corr_28, _, _ = multipletests(p_vals_28, alpha=0.05, method='fdr_bh')

# 初始化 8x8 全零(或全一)矩阵
t_stat_matrix = np.zeros((8, 8))
p_val_corrected_matrix = np.ones((8, 8))

# 将 28 个结果填回上三角
t_stat_matrix[upper_tri_indices] = t_vals_28
p_val_corrected_matrix[upper_tri_indices] = p_vals_corr_28

# 对称地填补至左下角（注意 T 值 A-B 等于 -(B-A)，而 P 值是对称相等的）
# 右上角存的是 row-col，左下角应该是 col-row = -(row-col)
t_stat_matrix.T[upper_tri_indices] = [-t for t in t_vals_28]
p_val_corrected_matrix.T[upper_tri_indices] = p_vals_corr_28

# 保存为 DataFrame
t_map_df = pd.DataFrame(t_stat_matrix, index=ordered_networks, columns=ordered_networks)
p_map_df = pd.DataFrame(p_val_corrected_matrix, index=ordered_networks, columns=ordered_networks)

t_map_df.to_csv(os.path.join(out_dir, 't_map_matrix_covariates.csv'))
p_map_df.to_csv(os.path.join(out_dir, 'p_map_corrected_matrix_covariates.csv'))
print("需求2已完成：控制年龄、性别、meanFD的组间比较已完成并保存。")

# ----------------- 5. 绘制 T-map 热图 -----------------

# ----------------- 5. 绘制 T-map 热图 -----------------
# 为了让热图更好看，我们创建一个带 '*' 的文本标注矩阵，p_corr < 0.05 的位置打星号
annot_matrix = np.empty((8, 8), dtype=object)
for i in range(8):
    for j in range(8):
        if i == j:
            annot_matrix[i, j] = ""  # 对角线留白
        else:
            t_val = t_stat_matrix[i, j]
            p_val = p_val_corrected_matrix[i, j]
            stars = "*" if p_val < 0.05 else ""  # 如果需要区分 0.01 也可以加 "**"
            # 格式：显示 T 值，如果显著再加个星号
            annot_matrix[i, j] = f"{t_val:.2f}{stars}"

plt.figure(figsize=(10, 8))

# --- 自定义热图颜色 ---
my_colors = ["#00478D", "#7DA2C6", "#FAFDFF", "#D78393", "#B50A27"]
custom_cmap = LinearSegmentedColormap.from_list("custom_diverging", my_colors)

# 绘制热图，应用自定义的 custom_cmap
ax = sns.heatmap(t_map_df, cmap=custom_cmap, center=0,
                 annot=annot_matrix, fmt='',
                 square=True, linewidths=.5,
                 #cbar_kws={"label": "T-value (Subtype1 vs Subtype2)"}
                 )

#plt.title('T-map of Network Differences\n(Controlling for Age, Sex, meanFD; * FDR $p<0.05$)')
plt.xticks(rotation=45, ha='right',fontsize=18)
plt.yticks(rotation=0,fontsize=18)

ax.tick_params(length=0)

vmin = t_map_df.min().min()
vmax = t_map_df.max().max()
cbar = ax.collections[0].colorbar
cbar.set_ticks([])
#cbar.set_ticklabels([f"{vmin:.2f}", f"{vmax:.2f}"])
cbar.ax.tick_params(length=0)
plt.tight_layout()

# 保存热图
heatmap_path = os.path.join(out_dir, 't_map_heatmap2.png')
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
print(f"需求3已完成：热图已保存至 {heatmap_path}")
plt.show()

需求1已完成：单被试的 28 个网络差异已拉平并与协变量合并保存。
需求2已完成：控制年龄、性别、meanFD的组间比较已完成并保存。
需求3已完成：热图已保存至 /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_FirstRe_covariate/Episode_new/Step5_DiffNetwork/t_map_heatmap2.png


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_86519/1981157332.py:158: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


QIDS 亚型间差异比较

In [12]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# 强制使用 Tkinter 后端弹窗，避免部分 IDE 报错 (如果您在 jupyter 运行，请注释掉此行)
matplotlib.use('TkAgg')

# ================= 配置参数区 =================
target_metric = 'QIDS'  # 分析的指标名称

basepath = '/Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_Match/Step2_subtype/Step6_QIDS_Diff/'
file_path1 = basepath + "/subtype1_QIDS.csv"
file_path2 = basepath + "/subtype2_QIDS.csv"

group_name1 = 'Subtype 1'
group_name2 = 'Subtype 2'
# ==============================================

# 1. 读取数据并打上组别标签
df1 = pd.read_csv(file_path1)
df2 = pd.read_csv(file_path2)

df1['Group'] = group_name1
df2['Group'] = group_name2

# 提取所需列并去除含有空值的行，确保统计模型能正常运行
df1 = df1.dropna(subset=[target_metric, 'age', 'sex'])
df2 = df2.dropna(subset=[target_metric, 'age', 'sex'])

orig_len1 = len(df1)
orig_len2 = len(df2)

# 2. 准备两份数据：原始数据 vs 剔除异常值数据 (IQR)
def get_clean_df(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

df1_clean = get_clean_df(df1, target_metric)
df2_clean = get_clean_df(df2, target_metric)

# 合并两份数据集
df_orig = pd.concat([df1, df2], ignore_index=True)
df_clean = pd.concat([df1_clean, df2_clean], ignore_index=True)

# 数据类型强转函数
def format_df(df):
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['sex'] = df['sex'].astype(str)
    df['Group'] = df['Group'].astype('category')
    return df

df_orig = format_df(df_orig)
df_clean = format_df(df_clean)

# 将两份数据放入字典，方便后续循环遍历出图
datasets = {
    "Original": df_orig,
    "Cleaned": df_clean
}

# 全局背景设置
sns.set_theme(style='white', rc={'axes.facecolor': 'white', 'figure.facecolor': 'white'})

custom_palette = {group_name1: '#B50A27', group_name2: '#00478D'}
groups_order = [group_name1, group_name2]
formula = f'{target_metric} ~ C(Group) + age + C(sex)'

# 遍历两份数据并分别单独出图
for data_type, current_df in datasets.items():

    # ---------------- 统计检验 (ANCOVA) ----------------
    model = smf.ols(formula, data=current_df).fit()
    anova_results = anova_lm(model, typ=2)
    p_val = anova_results.loc['C(Group)', 'PR(>F)']
    f_stat = anova_results.loc['C(Group)', 'F']

    print("="*50)
    print(f"当前分析状态: {data_type} Data")
    print(f"样本量: {group_name1}(n={len(current_df[current_df['Group']==group_name1])}) | {group_name2}(n={len(current_df[current_df['Group']==group_name2])})")
    print(f"F 统计量: {f_stat:.4f} | p 值: {p_val:.4f}")
    if p_val < 0.05:
        print("结论: 存在显著差异 (*p < 0.05)")
    else:
        print("结论: 没有显著差异 (p >= 0.05)")

    # ---------------- 每次循环新建一个独立画板 ----------------
    plt.figure(figsize=(8, 6))

    # 绘制箱线图
    ax = sns.boxplot(
        x='Group', y=target_metric, data=current_df,
        order=groups_order, palette=custom_palette, width=0.5,
        showfliers=False, linewidth=1.7,
        boxprops={'facecolor': 'none', 'alpha': 0.8}
    )

    # 核心风格改造：按组别给空心箱体和线条上色
    for i, group_name in enumerate(groups_order):
        group_color = custom_palette[group_name]

        if i < len(ax.patches):
            ax.patches[i].set_facecolor('none')
            ax.patches[i].set_edgecolor(group_color)
            ax.patches[i].set_linewidth(3)

        for j in range(5):
            line_idx = i * 5 + j
            if line_idx < len(ax.lines):
                ax.lines[line_idx].set_color(group_color)
                ax.lines[line_idx].set_linewidth(4)

    # 叠加散点图
    sns.stripplot(
        x='Group', y=target_metric, data=current_df,
        order=groups_order, palette=custom_palette, alpha=0.8,
        size=6, jitter=True
    )

    # 边框加粗
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(3.0)
        spine.set_visible(True)

    # ---------------- 绘制显著性星号 ----------------
    def get_stars(p):
        if p < 0.001: return '***'
        elif p < 0.01: return '**'
        elif p < 0.05: return '*'
        else: return 'ns'

    star_text = get_stars(p_val)

    y_max = current_df[target_metric].max()
    y_min = current_df[target_metric].min()
    y_range = y_max - y_min

    step = 0.08 * y_range
    h = 0.02 * y_range
    current_y = y_max + 0.05 * y_range

    def add_bracket(x1, x2, y, text_star):
        plt.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=2.5, color='black')
        plt.text((x1 + x2) * 0.5, y, text_star, ha='center', va='bottom', color='black', fontsize=18, fontweight='bold')

    # 仅当有显著差异时才画线
    if star_text != 'ns':
        add_bracket(0, 1, current_y, star_text)
        current_y += step

    # ---------------- 单独处理刻度与 Y 轴范围 ----------------
    plt.ylabel('')  # 清空 Y 轴文字
    plt.xlabel('')
    plt.xticks(fontsize=26)

    if data_type == "Original":
        # 对于 Original 图，修改特定的刻度 [10, 20, 30, 49]
        plt.yticks([10, 20, 30, 40], fontsize=26)
        adjusted_y_min = min(10, y_min - 0.05 * y_range)
        adjusted_y_max = max(50, current_y + 0.1 * y_range) # 留出空间防止 49 或连线被截取
        plt.ylim(adjusted_y_min, adjusted_y_max)
    else:
        # 对于 Cleaned 图，保持默认或常规逻辑
        plt.yticks([10, 15, 20,25],fontsize=26)
        adjusted_y_min = min(5, y_min - 0.05 * y_range)
        adjusted_y_max = max(20, current_y + 0.1 * y_range)
        plt.ylim(adjusted_y_min, adjusted_y_max)

    plt.tight_layout()

    # ---------------- 分别保存文件 ----------------
    save_filename = basepath + f"Fig_{target_metric}_{data_type}_ANCOVA_Comparison.png"
    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    print(f"✅ 图表已保存至: {save_filename}")

print("="*50)

# 最后统一展示弹窗 (会弹出两个独立窗口)
plt.show()

当前分析状态: Original Data
样本量: Subtype 1(n=123) | Subtype 2(n=143)
F 统计量: 4.5072 | p 值: 0.0347
结论: 存在显著差异 (*p < 0.05)


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_63874/2376358345.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_63874/2376358345.py:120: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


✅ 图表已保存至: /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_Match/Step2_subtype/Step6_QIDS_Diff/Fig_QIDS_Original_ANCOVA_Comparison.png
当前分析状态: Cleaned Data
样本量: Subtype 1(n=122) | Subtype 2(n=143)
F 统计量: 3.1652 | p 值: 0.0764
结论: 没有显著差异 (p >= 0.05)


/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_63874/2376358345.py:97: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_63874/2376358345.py:120: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(
/var/folders/dh/dmftg40j21j249nqf2z5v2nh0000gn/T/ipykernel_63874/2376358345.py:186: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


✅ 图表已保存至: /Volumes/QC/INT/INT_BN246_HC135BP_MDD135BP_DZIII/INT_Match/Step2_subtype/Step6_QIDS_Diff/Fig_QIDS_Cleaned_ANCOVA_Comparison.png
